### List of Libraries/Dependencies:
Requires < 30s to import all libraries.

In [140]:
# Document Converter
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import re
import unicodedata
import json
import logging
import time
from datetime import datetime
from collections.abc import Iterable
from collections import defaultdict
from itertools import product
from pathlib import Path

from docling_core.types.doc.base import ImageRefMode
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.document import ConversionResult
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption, HTMLFormatOption

# Schema Loader
# Cite the libraryyyyyyyyyyyyyyyyyyyyyyyyyyyyyy
from rdflib import RDF, RDFS, OWL, URIRef, Namespace, Literal, Dataset, Graph
from rdflib.namespace import XSD, split_uri
import rdflib
import os

# Hybrid Chunker
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY (DOCLING)
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY (HUGGINGFACE)
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from transformers import AutoTokenizer
from docling.document_converter import DocumentConverter
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any

# Chunk Processor
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import spacy
from spacy.symbols import VERB, AUX

# Triple Extractor
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import openai
import os

# Triple Extractor - Batch Processing
from concurrent.futures import ThreadPoolExecutor
import json
import time
from sentence_transformers import CrossEncoder, export_static_quantized_openvino_model
from optimum.intel import OVQuantizationConfig
import torch

# E-R Normalizer
# 1. MinHash LSH
from datasketch import MinHash, MinHashLSH

# E-R Normalizer
# 2. FAISS Indexing
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Graph Generator
import copy

# Graph Visualizer
from pyvis.network import Network
import streamlit.components.v1 as components

### Document Converter

In [11]:
# Cleans documents
def clean_text(text: str) -> str:
    if not text:
        return text

    # 1. Normalize Unicode (fixes odd composed characters)
    text = unicodedata.normalize("NFKC", text)

    # 2. Replace the following:
    replacements = {
        "\u00A0": " ",  # NBSP
        "\u202F": " ",  # narrow NBSP
        "\u202f": " ",  # narrow NBSP
        "\u2009": " ",  # thin space
        "\u2007": " ",  # figure space
        "\x00": "",     # null bytes
        "\ufffd": ""    # Unicode replacement char
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    # 4. Remove control characters (but keep newlines/tabs)
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", text)

    return text

_log = logging.getLogger(__name__)

# Export toggles:
# - USE_V2 controls modern Docling document exports.
# - USE_LEGACY enables legacy Deep Search exports for comparison or migration.
USE_V2 = True
USE_LEGACY = False


def export_documents(
    conv_results: Iterable[ConversionResult],
    output_dir: Path,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failure_count = 0
    partial_success_count = 0

    for conv_res in conv_results:
        if conv_res.status == ConversionStatus.SUCCESS:
            success_count += 1
            doc_filename = conv_res.input.file.stem

            if USE_V2:
                # Export converted files as markdown files
                conv_res.document.save_as_markdown(
                    output_dir / f"{doc_filename}.md",
                    image_mode=ImageRefMode.PLACEHOLDER,
                )
                
                # conv_res.document.save_as_json(
                #     output_dir / f"{doc_filename}.json",
                #     image_mode=ImageRefMode.PLACEHOLDER,
                # )
                
                # conv_res.document.save_as_markdown(
                #     output_dir / f"{doc_filename}.txt",
                #     image_mode=ImageRefMode.PLACEHOLDER,
                #     strict_text=True,
                # )

                # Export Docling document format to markdown:
                with (output_dir / f"{doc_filename}.md").open("w", encoding="utf-8") as fp:
                    raw_md = conv_res.document.export_to_markdown()
                    fp.write(clean_text(raw_md))

                # # Export Docling document format to text:
                # with (output_dir / f"{doc_filename}.txt").open("w") as fp:
                #     fp.write(conv_res.document.export_to_markdown(strict_text=True))

            if USE_LEGACY:
                # Export Markdown format:
                with (output_dir / f"{doc_filename}.legacy.md").open("w", encoding="utf-8") as fp:
                    raw_md = conv_res.document.export_to_markdown()
                    fp.write(clean_text(raw_md))

                # # Export Deep Search document JSON format:
                # with (output_dir / f"{doc_filename}.legacy.json").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(json.dumps(conv_res.document.export_to_dict()))

                # # Export Text format:
                # with (output_dir / f"{doc_filename}.legacy.txt").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(
                #         conv_res.document.export_to_markdown(strict_text=True)
                #     )

                # # Export Document Tags format:
                # with (output_dir / f"{doc_filename}.legacy.doctags.txt").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(conv_res.document.export_to_doctags())

        elif conv_res.status == ConversionStatus.PARTIAL_SUCCESS:
            _log.info(
                f"Document {conv_res.input.file} was partially converted with the following errors:"
            )
            for item in conv_res.errors:
                _log.info(f"\t{item.error_message}")
            partial_success_count += 1
        else:
            _log.info(f"Document {conv_res.input.file} failed to convert.")
            failure_count += 1

    _log.info(
        f"Processed {success_count + partial_success_count + failure_count} docs, "
        f"of which {failure_count} failed "
        f"and {partial_success_count} were partially converted."
    )
    return success_count, partial_success_count, failure_count


def main():
    logging.basicConfig(level=logging.INFO)

    # Location of source documents
    data_folder = Path("./eu_legislation")
    input_doc_paths = [file_path for file_path in data_folder.iterdir()]

    # buf = BytesIO((data_folder / "pdf/2206.01062.pdf").open("rb").read())
    # docs = [DocumentStream(name="my_doc.pdf", stream=buf)]
    # input = DocumentConversionInput.from_streams(docs)

    # # Turn on inline debug visualizations:
    # settings.debug.visualize_layout = True
    # settings.debug.visualize_ocr = True
    # settings.debug.visualize_tables = True
    # settings.debug.visualize_cells = True

    # Configure the PDF pipeline. Enabling page image generation improves HTML
    # previews (embedded images) but adds processing time.
    pdf_pipeline_options = PdfPipelineOptions()
    pdf_pipeline_options.generate_page_images = True

    doc_converter = DocumentConverter(
        allowed_formats=[
            InputFormat.PDF,
            InputFormat.HTML
        ],
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pdf_pipeline_options,
                backend=DoclingParseV4DocumentBackend
            ),
            InputFormat.HTML: HTMLFormatOption()
        }
    )

    start_time = time.time()

    # Convert all inputs. Set `raises_on_error=False` to keep processing other
    # files even if one fails; errors are summarized after the run.
    conv_results = doc_converter.convert_all(
        input_doc_paths,
        raises_on_error=False,  # to let conversion run through all and examine results at the end
    )
    # Write outputs to ./scratch and log a summary.
    _success_count, _partial_success_count, failure_count = export_documents(
        conv_results, output_dir=Path("converted_docs")
    )

    end_time = time.time() - start_time

    _log.info(f"Document conversion complete in {end_time:.2f} seconds.")

    if failure_count > 0:
        raise RuntimeError(
            f"The example failed converting {failure_count} on {len(input_doc_paths)}."
        )

if __name__ == "__main__":
    main()

2026-07-19 03:46:35,814 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2026-07-19 03:46:35,819 - INFO - Going to convert document batch...
2026-07-19 03:46:35,820 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-07-19 03:46:35,821 - INFO - Processing document 31953D0030en.html
2026-07-19 03:46:35,826 - INFO - Finished converting document 31953D0030en.html in 0.02 sec.
2026-07-19 03:46:35,833 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2026-07-19 03:46:35,839 - INFO - Going to convert document batch...
2026-07-19 03:46:35,840 - INFO - Processing document 31954S0024en.html
2026-07-19 03:46:35,842 - INFO - Finished converting document 31954S0024en.html in 0.01 sec.
2026-07-19 03:46:35,854 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2026-07-19 03:46:35,861 - INFO - Going to convert document batch...
2026-07-19 03:46:35,863 - INFO - Processing document 31954S0026en.html
2026-07-19 03:46:35,869 - INFO -

### Schema Loader

In [12]:
# Define your own custom namespaces for your knowledge graph:

JS_DATA = Namespace(
    "http://jurisynth/data/"
)

JS_SOURCE = Namespace(
    "http://jurisynth/source/"
)

In [13]:
schema_folder = "./schema"
schema_files = [os.path.join(schema_folder, file) for file in os.listdir(schema_folder)]

schema_files

['./schema\\cdm.rdf', './schema\\cdm_datatypes.rdf']

In [14]:
def extract_resources(graph, resource_type):
    """
    Extract resources of a given RDF type.

    Returns:
        resources: URI -> label
        metadata: URI -> metadata dict
    """

    resources = {}
    metadata = {}

    resource_types = {
        "class": OWL.Class,
        "object property": OWL.ObjectProperty,
        "datatype property": OWL.DatatypeProperty,
        "datatype": RDFS.Datatype
    }

    chosen_type = resource_types[resource_type]

    deprecated_count = 0

    for uri in set(graph.subjects(RDF.type, chosen_type)):

        if isinstance(uri, rdflib.term.BNode):
            continue

        label = graph.value(uri, RDFS.label)
        comment = graph.value(uri, RDFS.comment)

        label = (
            str(label)
            if label
            else uri.split("#")[-1]
        )

        comment = (
            str(comment)
            if comment
            else ""
        )

        # --------------------------------------------------
        # Detect deprecated resources
        # --------------------------------------------------

        is_deprecated = False

        deprecated = graph.value(uri, OWL.deprecated)

        if deprecated is not None:
            if str(deprecated).casefold() == "true":
                is_deprecated = True

        if "deprecated" in label.casefold():
            is_deprecated = True

        if "deprecated" in comment.casefold():
            is_deprecated = True

        if is_deprecated:
            deprecated_count += 1
            continue

        # --------------------------------------------------
        # Build NLP-friendly resource description
        # --------------------------------------------------

        text = label

        if comment:
            text += ". " + comment

        resources[str(uri)] = label

        metadata[str(uri)] = {
            "type": resource_type,
            "label": label,
            "comment": comment,
            "text": text,
            "domain": (
                set(graph.objects(uri, RDFS.domain))
                if resource_type in {"object property", "datatype property"}
                else set()
            ),
            "range": (
                set(graph.objects(uri, RDFS.range))
                if resource_type in {"object property", "datatype property"}
                else set()
            )
        }

    print(
        f"Filtered {deprecated_count} deprecated "
        f"{resource_type.replace('_', ' ')} resources."
    )

    return resources, metadata

In [15]:
schema_graph = Graph()

index = 1
for file in schema_files:
    schema_graph.parse(file, format="xml")
    print(f"Finished parsing RDF file {index}.")
    index += 1

print()

# Only retrieves declared namespaces
schema_namespaces = set(schema_graph.namespaces())
# schema_namespaces.remove(('', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdmplus#')))
# schema_namespaces.add(('cdmplus#', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdmplus#')))

print("Schema Namespaces:")
for ns in schema_namespaces:
    print(ns)

print()

# Classes
classes, class_metadata = extract_resources(schema_graph, "class")
print(f"Found {len(classes)} classes.")

# Object properties
obj_properties, obj_metadata = extract_resources(schema_graph, "object property")
print(f"Found {len(obj_properties)} object properties/relations.")

# Datatype properties
datatype_properties, data_prop_metadata = extract_resources(schema_graph, "datatype property")
print(f"Found {len(datatype_properties)} datatype properties/relations.")

# Datatypes
datatypes, datatype_metadata = extract_resources(schema_graph, "datatype")
print(f"Found {len(datatypes)} datatypes.")

# Resource Metadata
resource_metadata = dict()

resource_metadata.update(class_metadata)
resource_metadata.update(obj_metadata)
resource_metadata.update(data_prop_metadata)
resource_metadata.update(datatype_metadata)


Finished parsing RDF file 1.
Finished parsing RDF file 2.

Schema Namespaces:
('schema', rdflib.term.URIRef('https://schema.org/'))
('dcam', rdflib.term.URIRef('http://purl.org/dc/dcam/'))
('prof', rdflib.term.URIRef('http://www.w3.org/ns/dx/prof/'))
('vann', rdflib.term.URIRef('http://purl.org/vocab/vann/'))
('dcat', rdflib.term.URIRef('http://www.w3.org/ns/dcat#'))
('annotation', rdflib.term.URIRef('http://publications.europa.eu/ontology/annotation#'))
('prov', rdflib.term.URIRef('http://www.w3.org/ns/prov#'))
('csvw', rdflib.term.URIRef('http://www.w3.org/ns/csvw#'))
('org', rdflib.term.URIRef('http://www.w3.org/ns/org#'))
('ssn', rdflib.term.URIRef('http://www.w3.org/ns/ssn/'))
('brick', rdflib.term.URIRef('https://brickschema.org/schema/Brick#'))
('dcmitype', rdflib.term.URIRef('http://purl.org/dc/dcmitype/'))
('sosa', rdflib.term.URIRef('http://www.w3.org/ns/sosa/'))
('sh', rdflib.term.URIRef('http://www.w3.org/ns/shacl#'))
('rdfs', rdflib.term.URIRef('http://www.w3.org/2000/01/r

In [16]:
resource_metadata

{'http://publications.europa.eu/ontology/cdm#act_standing-committee_efta': {'type': 'class',
  'label': 'Act of EFTA Standing Committee',
  'comment': 'Act of European Free Trade Association (EFTA) Standing Committee (= subsector G of sector E in EUR-Lex)',
  'text': 'Act of EFTA Standing Committee. Act of European Free Trade Association (EFTA) Standing Committee (= subsector G of sector E in EUR-Lex)',
  'domain': set(),
  'range': set()},
 'http://publications.europa.eu/ontology/cdm#opinion-reasoned-np-eums': {'type': 'class',
  'label': 'Reasoned opinion of EU member-state national parliament',
  'comment': 'The reasoned opinion of an EU member-state national parliament as reaction on the proposal of the Commission in scope of an interinstitutional procedure',
  'text': 'Reasoned opinion of EU member-state national parliament. The reasoned opinion of an EU member-state national parliament as reaction on the proposal of the Commission in scope of an interinstitutional procedure',
  '

#### Resource-URI dictionaries

In [17]:
rdf_resources = list(classes.keys()) + list(obj_properties.keys()) + list(datatype_properties.keys()) + list(datatypes.keys())
rdf_dict = {**classes, **obj_properties, **datatype_properties, **datatypes}

##### *Inspecting the Retrieved Classes:*

In [18]:
list(classes.items())

[('http://publications.europa.eu/ontology/cdm#act_standing-committee_efta',
  'Act of EFTA Standing Committee'),
 ('http://publications.europa.eu/ontology/cdm#opinion-reasoned-np-eums',
  'Reasoned opinion of EU member-state national parliament'),
 ('http://publications.europa.eu/ontology/cdm#periodical_work',
  'Periodical Work'),
 ('http://publications.europa.eu/ontology/cdm#opinion-political-dialogue-np-eums',
  'Opinion political dialogue of EU member-state national parliament'),
 ('http://publications.europa.eu/ontology/cdm#manifestation_case-law',
  'Manifestation of case law'),
 ('http://publications.europa.eu/ontology/cdm#committee', 'Committee'),
 ('http://publications.europa.eu/ontology/cdm#act_preparatory',
  'Preparatory act'),
 ('http://publications.europa.eu/ontology/cdm#restriction_on_use',
  'Restriction on use'),
 ('http://publications.europa.eu/ontology/cdm#opinion_eesc',
  'European Economic and Social Committee opinion'),
 ('http://publications.europa.eu/ontology/cd

In [19]:
print("No. of RDF Resources:", len(rdf_resources))

No. of RDF Resources: 2625


### Hybrid Chunker

In [20]:
EMBED_MODEL_ID = "openai/gpt-oss-120b"

MAX_TOKENS = 1024

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=MAX_TOKENS,
)

chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,
)

digitalizer = DocumentConverter()

conv_docs_folder = Path("./converted_docs")
conv_doc_paths = [file_path for file_path in conv_docs_folder.iterdir()]

raw_chunks = list()

for doc_path in conv_doc_paths:

    filename = os.path.basename(doc_path)
    doc = digitalizer.convert(source=doc_path).document
    chunks = chunker.chunk(dl_doc=doc)

    for i, chunk in enumerate(chunks):
        raw_chunks.append(
            {
                "doc_id": filename,
                "chunk_id": f"chunk_{i + 1}",
                "chunk": chunk
            }
        )

2026-07-19 03:46:51,546 - INFO - HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-07-19 03:46:51,856 - INFO - HTTP Request: HEAD https://huggingface.co/openai/gpt-oss-120b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-19 03:46:51,871 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/gpt-oss-120b/b5c939de8f754692c1647ca79fbf85e8c1e70f8a/config.json "HTTP/1.1 200 OK"
2026-07-19 03:46:52,168 - INFO - HTTP Request: HEAD https://huggingface.co/openai/gpt-oss-120b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-19 03:46:52,187 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/gpt-oss-120b/b5c939de8f754692c1647ca79fbf85e8c1e70f8a/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-19 03:46:52,459 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/gpt-oss-120b/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.

##### *Chunk Inspection:*

In [21]:
# Pick the first document
sample_file = raw_chunks[0]["doc_id"]

# Get all chunks belonging to that document
sample_chunk_list = [
    item
    for item in raw_chunks
    if item["doc_id"] == sample_file
]

print("The current chunks are from:", sample_file, "\n")

for item in sample_chunk_list:
    chunk_id = item["chunk_id"]
    chunk = item["chunk"]

    print(f"=== {chunk_id} ===")

    txt_tokens = tokenizer.count_tokens(chunk.text)
    print(f"chunk.text ({txt_tokens} tokens):\n{chunk.text!r}")

    ser_txt = chunker.contextualize(chunk=chunk)
    ser_tokens = tokenizer.count_tokens(ser_txt)
    print(
        f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_txt!r}"
    )

    print()

The current chunks are from: 31953D0030en.md 

=== chunk_1 ===
chunk.text (303 tokens):
"**ECSC High Authority: Decision No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel**\n*Official Journal 006 , 04/05/1953 P. 0109 - 0110 Danish special edition: Series I Chapter 1952-1958 P. 0009 English special edition: Series I Chapter 1952-1958 P. 0009 Greek special edition: Chapter 08 Volume 1 P. 0005 Spanish special edition: Chapter 08 Volume 1 P. 0005 Portuguese special edition Chapter 08 Volume 1 P. 0005 Finnish special edition: Chapter 12 Volume 3 P. 0003 Swedish special edition: Chapter 12 Volume 3 P. 0003*\nDECISION No 30-53  of 2 May 1953  on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel THE HIGH AUTHORITY, Having regard to Article 60 and Article 63 (2) of the Treaty; Whereas compliance with the obligations of non-discrimination involves uniform application by undertakings of

### Chunk Processor

In [23]:
nlp = spacy.load("en_core_web_lg")

def strip_all_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

# ---------------------------------------------------------------------
# Batch chunks
# ---------------------------------------------------------------------

docs = nlp.pipe(
    (item["chunk"].text for item in raw_chunks),
    batch_size=64,
    n_process=2
)

# ---------------------------------------------------------------------
# Process chunks
# ---------------------------------------------------------------------

processed_chunks = list()

for item, chunk in zip(raw_chunks, docs):

    filename = item["doc_id"]
    chunk_id = item["chunk_id"]

    filtered_sents = list()

    for sent in chunk.sents:
        text = sent.text.strip()

        if not text:
            continue

        # check for VERB or AUX in the sentence
        has_verb_or_aux = any(
            token.pos_ in {"VERB", "AUX"} for token in sent
        )

        if not has_verb_or_aux:
            continue

        filtered_sents.append(text)

    if not filtered_sents:
        continue

    processed_chunk = " ".join(filtered_sents)

    cleaned = strip_all_whitespace(processed_chunk)

    if cleaned:
        processed_chunks.append({
            "doc_id": filename,
            "chunk_id": chunk_id,
            "content": cleaned
        })

    print("CHUNK:", repr(cleaned))
    print("FILE:", filename, "\n")

CHUNK: "No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel** *Official Journal 006 , 04/05/1953 P. 0109 - 0110 Danish special edition: Series I Chapter 1952-1958 P. 0009 Finnish special edition: Chapter 12 Volume 3 P. 0003 Swedish special edition: Chapter 12 Volume 3 P. 0003* DECISION No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel THE HIGH AUTHORITY, Having regard to Article 60 and Article 63 (2) of the Treaty; Whereas compliance with the obligations of non-discrimination involves uniform application by undertakings of the conditions shown in their price lists with no other increases or reductions and no evasion of those obligations by allowing longer periods for settlement without a corresponding increase in price; Whereas the exception to this rule, namely the option of aligning a quotation on a competitor's price list,"
FILE: 31953D0030en.md 

C

### Triple Extractor

In [82]:
# Security Measure
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')

client = openai.OpenAI(
    base_url = "https://integrate.api.nvidia.com/v1",
    api_key = openai.api_key
    )

extraction_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {

            "subject": {
                "type": "string",
                "minLength": 1
            },

            "predicate": {
                "type": "string",
                "minLength": 1
            },

            "object": {
                "type": "string",
                "minLength": 1
            },
        },

        "required": [
            "subject",
            "predicate",
            "object"
        ],

        "additionalProperties": False
    }
}

extraction_prompt = """
You will be given a chunk of text from an EU legal document.

Extract legal semantic triples as JSON.

## Extraction rules

1. Extract only facts that are explicitly stated or unambiguously defined in the text.

2. Prioritize precision over recall.
   If a relationship is unclear, omit the triple rather than guessing.

3. Resolve pronouns, demonstratives, and references to their actual entities.

4. Subjects and objects must represent meaningful entities, concepts, actions, or values from the text.
   Do not use:
   - boolean values ("true", "false")
   - vague placeholders ("it", "this", "the above")
   - entire sentences as objects unless they represent a specific condition or concept.

5. Use "type of" only when the text explicitly defines an entity's category.

   Valid examples:
   - (Regulation (EU) 2024/1234, type of, Regulation)
   - (Article 5, type of, Article)
   - (European Commission, type of, Institution)

6. Preserve the original meaning of legal predicates.
   Keep negations and polarity:
   - "does not apply" is different from "applies"
   - "is prohibited" is different from "is permitted"

7. Ignore:
   - document formatting
   - pagination
   - headers and footers
   - publication metadata
   - editorial information
   - chapter/section numbering unless it is explicitly referenced as a legal entity.

"""

def get_completion(
        system_prompt="",
        query="",
        schema=extraction_schema
        ):
    
    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
           {'role':'system', 'content': system_prompt},
           {'role':'user', 'content': query}
           ],
        temperature=0,
        top_p=0.1, # Test different values
        max_tokens=None,
        stream=False,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "triples",
                "strict": True,
                "schema": schema
                }
            }
        )

    return completion.choices[0].message.content


#### Batch Processing

In [83]:
extraction_results = list()

# ---------------------------------------------------------------------
# Triple extraction
# ---------------------------------------------------------------------

def extraction_thread(chunk_dict):
    response = get_completion(
        system_prompt=extraction_prompt,
        query=chunk_dict["content"],
        schema=extraction_schema
    )

    result = json.loads(response)

    return {
        "doc_id": chunk_dict["doc_id"],
        "chunk_id": chunk_dict["chunk_id"],
        "triples": result
    }


with ThreadPoolExecutor(max_workers=10) as executor:
    extraction_results = list(
        executor.map(
            extraction_thread,
            processed_chunks
        )
    )


# ---------------------------------------------------------------------
# Flatten extracted triples
# ---------------------------------------------------------------------

extracted_triples = list()

for result in extraction_results:

    doc_id = result["doc_id"]
    chunk_id = result["chunk_id"]
    triples = result["triples"]

    if isinstance(triples, str):
        triples = [triples]

    elif not isinstance(triples, list):
        continue

    extracted_triples.append(
        {
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "triples": triples
        }
    )

2026-07-19 05:46:28,339 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 05:46:28,443 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 05:46:30,796 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 05:46:32,907 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 05:46:35,316 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 05:46:36,343 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 05:46:36,803 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 05:46:37,458 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 05:46

In [203]:
nlp = spacy.load(
    "en_core_web_sm",
    disable=["ner"]
)

# ---------------------------------------------------------------------
# Normalization
# ---------------------------------------------------------------------

def normalize_entity(text: str) -> str:
    """
    Conservative cleanup of entity names.
    Preserves legal wording.
    """

    text = text.strip()

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove trailing punctuation
    text = text.rstrip(".,;:")

    return text


def normalize_predicate(text: str) -> str:
    """
    Cleanup and canonicalize predicate phrases.
    """

    text = re.sub(r"\s+", " ", text.strip())

    replacements = {
        "is obliged to": "obligated to",
        "is required to": "required to",
        "is subject to": "subject to",
        "is entitled to": "entitled to",
    }

    lower = text.lower()

    for old, new in replacements.items():
        if lower.startswith(old):
            text = new + text[len(old):]
            break

    return text


# ---------------------------------------------------------------------
# Legal reference splitting
# ---------------------------------------------------------------------

def split_legal_reference(text: str):

    pattern = (
        r"\s+(?:and|or)\s+"
        r"(?=(?:article|paragraph|point|section|annex)\b)"
    )

    parts = re.split(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if len(parts) > 1:
        return [
            part.strip()
            for part in parts
        ]

    return None


# ---------------------------------------------------------------------
# Cheap composite detection heuristic
# ---------------------------------------------------------------------

def should_attempt_split(text: str):
    """
    Avoid unnecessary spaCy parsing.

    Only attempt NLP splitting when the text
    contains likely coordination markers.
    """

    indicators = [
        " and ",
        " or ",
        ",",
        "Article ",
        "Paragraph ",
        "Point ",
        "Section ",
        "Annex "
    ]

    text_lower = text.lower()

    return any(
        indicator.lower() in text_lower
        for indicator in indicators
    )


# ---------------------------------------------------------------------
# Composite entity detection
# ---------------------------------------------------------------------

def split_entity(text: str):
    """
    Returns:
        None              -> no safe expansion
        list[str]         -> safe expansion
    """

    # First handle explicit legal references
    legal_split = split_legal_reference(text)

    if legal_split:
        return legal_split


    # Skip expensive NLP when unlikely to be composite
    if not should_attempt_split(text):
        return None


    # Fall back to dependency parsing
    doc = nlp(text)


    # Require coordination
    if not any(
        tok.dep_ == "conj"
        for tok in doc
    ):
        return None


    # Avoid complex clauses
    if any(
        tok.dep_ in {
            "relcl",
            "advcl",
            "ccomp",
            "xcomp"
        }
        for tok in doc
    ):
        return None


    noun_chunks = list(doc.noun_chunks)

    if len(noun_chunks) < 2:
        return None

    entities = [
        chunk.text.strip()
        for chunk in noun_chunks
    ]


    entities = list(dict.fromkeys(entities))

    return (
        entities
        if len(entities) > 1
        else None
    )


# ---------------------------------------------------------------------
# Triple expansion
# ---------------------------------------------------------------------

def expand_triple(triple: dict):

    subject = normalize_entity(triple["subject"])
    predicate = normalize_predicate(triple["predicate"])
    object = normalize_entity(triple["object"])

    subjects = (
        split_entity(subject)
        or [subject]
    )

    objects = (
        split_entity(object)
        or [object]
    )

    expanded = list()


    # Avoid unnecessary Cartesian expansion
    if (
        len(subjects) == len(objects)
        and len(subjects) > 1
    ):
        pairs = zip(subjects, objects)

    else:
        pairs = product(subjects, objects)


    for s, o in pairs:
        expanded.append({
            "subject": s,
            "predicate": predicate,
            "object": o
            })

    return expanded


# ---------------------------------------------------------------------
# Batch normalization
# ---------------------------------------------------------------------

def normalize_triples(extracted_triples):
    """
    Normalize and expand extracted triples.

    Input:
        [
            {
                "doc_id": ...,
                "chunk_id": ...,
                "triples": [...]
            },
            ...
        ]

    Output:
        [
            {
                "doc_id": ...,
                "chunk_id": ...,
                "triple": {...}
            },
            ...
        ]
    """

    normalized = list()

    required_fields = {
        "subject",
        "predicate",
        "object"
    }

    for chunk in extracted_triples:

        doc_id = chunk["doc_id"]
        chunk_id = chunk["chunk_id"]

        for triple in chunk["triples"]:

            # Skip malformed triples
            if not isinstance(triple, dict):
                continue

            if not required_fields.issubset(triple):
                continue

            expanded = expand_triple(triple)

            for tr in expanded:

                normalized.append(
                    {
                        "doc_id": doc_id,
                        "chunk_id": chunk_id,
                        "triple_id": len(normalized),
                        "triple": tr
                    }
                )

    return normalized

In [204]:
normalized_triples = normalize_triples(extracted_triples)

### Semantic Filter

In [105]:
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

resources = {
    "class": classes,
    "obj_prop": obj_properties,
    "datatype_prop": datatype_properties,
    "datatype": datatypes,
}

index_lookup = dict()

for resource_type, resource_dict in resources.items():
    
    texts = [
        resource_metadata[uri]["text"]
        for uri in resource_dict.keys()
    ]
    
    uris = list(resource_dict.keys())

    embeddings = emb_model.encode(
        texts,
        normalize_embeddings=True
    )

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    index_lookup[resource_type] = (index, uris, texts)

2026-07-19 06:18:50,187 - INFO - No device provided, using cpu
2026-07-19 06:18:50,616 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-19 06:18:50,638 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-19 06:18:51,004 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-19 06:18:51,027 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-19 06:18:51,031 - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-07-19 06

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-19 06:18:53,158 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-19 06:18:53,436 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-19 06:18:53,701 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-19 06:18:53,985 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-19 06:18:54,316 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-19 06:18:54,340 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/senten

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [205]:
TYPE_MAP = {
    "string": XSD.string,
    "integer": XSD.integer,
    "decimal": XSD.decimal,
    "boolean": XSD.boolean,
    "date": XSD.date,
    "datetime": XSD.dateTime,
}

DATE_FORMATS = [
    "%Y-%m-%d",

    # Common EU/legal formats
    "%d-%m-%Y",
    "%d/%m/%Y",
    "%d.%m.%Y",

    # Written dates
    "%d %B %Y",
    "%d %b %Y",
]

DATETIME_FORMATS = [
    "%Y-%m-%dT%H:%M:%S",
    "%Y-%m-%dT%H:%M:%SZ",
    "%Y-%m-%dT%H:%M:%S%z",

    # ISO with fractional seconds
    "%Y-%m-%dT%H:%M:%S.%f",
    "%Y-%m-%dT%H:%M:%S.%fZ",
    "%Y-%m-%dT%H:%M:%S.%f%z",
]

LEGAL_IDENTIFIER_PATTERNS = [

    # Article / paragraph references
    r"\b(article|art\.?)\s*\d+(\s*\([a-z0-9]+\))?",

    # EU legal acts with numbers
    # Regulation No 1395/69
    # Decision 68/302/EEC
    # Directive 2006/123/EC
    r"\b(regulation|directive|decision|recommendation|opinion)"
    r".*\b(no\.?|number)?\s*\d+[-/]\d+",

    # EU identifiers:
    # 2006/123/EC
    # 68/302/EEC
    r"\b\d{1,4}/\d{1,4}/[a-z]{2,5}\b",

    # CELEX-like identifiers
    r"\b\d{5}[A-Z]\d{4}\b",
]

NEGATION_WORDS = {
    "not",
    "no",
    "never",
    "without",
    "neither",
    "nor",
    "n't"
}

def detect_literal_type(value: str):

    v = clean_text(str(value)).strip()

    # boolean
    if v.lower() in {"true", "false"}:
        return "boolean"

    # integer
    if re.fullmatch(r"-?\d+", v):
        return "integer"

    # decimal
    if re.fullmatch(r"-?\d+\.\d+", v):
        return "decimal"

    # datetime
    for fmt in DATETIME_FORMATS:
        try:
            datetime.strptime(v, fmt)
            return "datetime"
        except ValueError:
            pass

    # date
    for fmt in DATE_FORMATS:
        try:
            datetime.strptime(v, fmt)
            return "date"
        except ValueError:
            pass

    return "class"

def normalize_date(value: str, literal_type: str):
    # value is expected to already be cleaned by the caller
    if literal_type == "date":
        for fmt in DATE_FORMATS:
            try:
                dt = datetime.strptime(value, fmt)
                return dt.strftime("%Y-%m-%d")
            except ValueError:
                continue

    elif literal_type == "datetime":
        for fmt in DATETIME_FORMATS:
            try:
                dt = datetime.strptime(value, fmt)
                return dt.isoformat()
            except ValueError:
                continue

    return value

def is_legal_identifier_reference(text: str) -> bool:
    """
    Detect whether a label contains a likely legal identifier.

    This is a heuristic filter only.
    """

    text = text.casefold()

    for pattern in LEGAL_IDENTIFIER_PATTERNS:
        if re.search(pattern, text):
            return True

    return False

def extract_polarity_doc(doc):
    """
    Returns True for affirmative predicates,
    False for explicitly negative predicates.

    Designed for short ontology labels rather than
    full natural-language sentences.
    """

    for token in doc:

        if token.lower_ in NEGATION_WORDS:
            return False

    return True

def extract_polarity(text):
    """
    Convenience wrapper when a Doc is unavailable.
    """

    return extract_polarity_doc(
        nlp.make_doc(text)
    )

def create_custom_uri(text: str, prefix=JS_DATA):
    """
    Generate a stable URI for unresolved components.

    Applies conservative normalization only.
    """

    text = text.casefold()

    # Unicode normalization
    text = unicodedata.normalize(
        "NFKC",
        text
    )

    # Replace whitespace with underscores
    text = re.sub(
        r"\s+",
        "_",
        text
    )

    # Remove unsafe URI characters
    text = re.sub(
        r"[^a-z0-9_\-]",
        "",
        text
    )

    return URIRef(
        prefix[text]
    )


# --------------------------------------------------
# 1. Collect lookup requests
# --------------------------------------------------

lookup_requests = defaultdict(list)
scored_triples = copy.deepcopy(normalized_triples)

for idx, triple in enumerate(normalized_triples):
    triple = normalized_triples[idx]["triple"]
    scored_triple = scored_triples[idx]["triple"]

    # Subject -> always class lookup for now
    if is_legal_identifier_reference(triple["subject"]):
        scored_triple["subject"] = create_custom_uri(triple["subject"])

    else:
        lookup_requests["class"].append({
            "idx": idx,
            "field": "subject",
            "text": str(triple["subject"])
        })

    # Object
    object_type = detect_literal_type(triple["object"])

    if object_type == "class":

        if is_legal_identifier_reference(triple["object"]):
            scored_triple["object"] = create_custom_uri(triple["object"])
        
        else:
            lookup_requests["class"].append({
                "idx": idx,
                "field": "object",
                "text": str(triple["object"])
            })


    else:
        clean_value = clean_text(str(triple["object"])).strip()

        if object_type in {"date", "datetime"}:
            value = normalize_date(clean_value, object_type)
        elif object_type == "integer":
            value = int(clean_value)
        elif object_type == "decimal":
            value = float(clean_value)
        else:
            value = scored_triple["object"]

        lit = Literal(value, datatype=TYPE_MAP[object_type])

        if object_type == "date":
            try:
                rdflib.xsd_datetime.parse_xsd_date(str(lit))
            except Exception as e:
                raise ValueError(
                    f"Bad date literal at idx={idx}: raw={triple['object']!r} "
                    f"clean={clean_value!r} normalized={value!r}"
                ) from e

        scored_triple["object"] = lit

    # Predicate
    if triple["predicate"] == "type of":
        scored_triple["predicate"] = RDF.type

    else:
        prop_type = (
            "obj_prop"
            if object_type == "class"
            else "datatype_prop"
        )

        lookup_requests[prop_type].append({
            "idx": idx,
            "field": "predicate",
            "text": str(triple["predicate"])
        })


# --------------------------------------------------
# 2. Batch semantic lookup
# --------------------------------------------------

lookup_results = dict()

# Cache polarity across all predicate lookups
polarity_cache = dict()

for resource_type, requests in lookup_requests.items():

    texts = [
        r["text"]
        for r in requests
    ]

    # --------------------------------------------------
    # Sentence embeddings
    # --------------------------------------------------

    embeddings = emb_model.encode(
        texts,
        batch_size=128,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    index, uris, labels = index_lookup[resource_type]

    scores, indices = index.search(embeddings, 3)

    # --------------------------------------------------
    # Candidate construction
    # --------------------------------------------------

    for req, score_row, idx_row in zip(
        requests,
        scores,
        indices
    ):

        candidates = [
            (
                labels[i],
                uris[i],
                float(score_row[j])
            )
            for j, i in enumerate(idx_row)
        ]


        # --------------------------------------------------
        # Predicate polarity filtering
        # --------------------------------------------------

        if req["field"] == "predicate":
            query_text = req["text"]

            if query_text not in polarity_cache:

                polarity_cache[query_text] = extract_polarity(
                    query_text
                )

            query_polarity = polarity_cache[query_text]
            filtered_candidates = list()

            for candidate in candidates:
                candidate_label = candidate[0]

                if candidate_label not in polarity_cache:

                    polarity_cache[candidate_label] = extract_polarity(
                        candidate_label
                    )

                if (
                    polarity_cache[candidate_label]
                    == query_polarity
                ):
                    filtered_candidates.append(candidate)

            # Only apply filtering if at least one candidate survives
            if filtered_candidates:
                candidates = filtered_candidates

        lookup_results[
            (
                req["idx"],
                req["field"],
                req["text"]
            )
        ] = candidates


# --------------------------------------------------
# 3. Resource Matching/Filtering
# --------------------------------------------------

custom_resources = dict()

THRESHOLD = 0.7

for (idx, field, text), candidates in lookup_results.items():

    scored_triple = scored_triples[idx]["triple"]

    best = candidates[0]
    score = best[2]

    if score >= THRESHOLD:
        # High-confidence semantic match
        scored_triple[field] = URIRef(best[1])

    else:
        # Assign custom URI
        custom_uri = create_custom_uri(text)
        scored_triple[field] = custom_uri
        custom_resources.setdefault(
            custom_uri,
            {
                "text": text,
                "field": field,
                "candidates": candidates
            }
        )

Batches:   0%|          | 0/145 [00:00<?, ?it/s]

Batches:   0%|          | 0/76 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [206]:
full_list = [
    (component_name)
    for element in scored_triples
    for component_name in element["triple"]
    ]

inspection_list = [
    (component_name, component_value, element["triple"])
    for element in scored_triples
    for component_name, component_value in element["triple"].items()
    if str(component_value).startswith("http://publications")
]

In [207]:
print(len(full_list), len(inspection_list))

31032 1744


In [208]:
scored_triples[50:100]

[{'doc_id': '31953D0030en.md',
  'chunk_id': 'chunk_2',
  'triple_id': 50,
  'triple': {'subject': rdflib.term.URIRef('http://jurisynth/data/conditions'),
   'predicate': rdflib.term.URIRef('http://jurisynth/data/is_permitted_not_prohibited_under'),
   'object': rdflib.term.URIRef('http://jurisynth/data/article_601_of_the_treaty')}},
 {'doc_id': '31953D0030en.md',
  'chunk_id': 'chunk_2',
  'triple_id': 51,
  'triple': {'subject': rdflib.term.URIRef('http://jurisynth/data/sale'),
   'predicate': rdflib.term.URIRef('http://jurisynth/data/is_permitted_not_prohibited_under'),
   'object': rdflib.term.URIRef('http://jurisynth/data/article_601_of_the_treaty')}},
 {'doc_id': '31953D0030en.md',
  'chunk_id': 'chunk_2',
  'triple_id': 52,
  'triple': {'subject': rdflib.term.URIRef('http://jurisynth/data/the_value'),
   'predicate': rdflib.term.URIRef('http://jurisynth/data/is_permitted_not_prohibited_under'),
   'object': rdflib.term.URIRef('http://jurisynth/data/article_601_of_the_treaty')}},

In [209]:
for name, value, triple in inspection_list[:100]:
    print(value)
    
    print(triple["subject"])
    print(triple["predicate"])
    print(repr(triple["object"]))
    
    print()


http://publications.europa.eu/ontology/cdm#includes
http://jurisynth/data/uniform_application_by_undertakings_of_conditions_shown_in_price_lists
http://publications.europa.eu/ontology/cdm#includes
rdflib.term.URIRef('http://jurisynth/data/no_other_increases')

http://publications.europa.eu/ontology/cdm#includes
http://jurisynth/data/uniform_application_by_undertakings_of_conditions_shown_in_price_lists
http://publications.europa.eu/ontology/cdm#includes
rdflib.term.URIRef('http://jurisynth/data/reductions')

http://publications.europa.eu/ontology/cdm#includes
http://jurisynth/data/uniform_application_by_undertakings_of_conditions_shown_in_price_lists
http://publications.europa.eu/ontology/cdm#includes
rdflib.term.URIRef('http://jurisynth/data/no_evasion_of_obligations_by_allowing_longer_periods_for_settlement_without_a_corresponding_increase_in_price')

http://publications.europa.eu/ontology/cdm#procurement_public
http://publications.europa.eu/ontology/cdm#procurement_public
http://jur

### Entity-Relation Resolver

In [213]:
triple_lookup = {
    element["triple_id"]: element["triple"]
    for element in scored_triples
}

In [214]:
def normalize_key(label: str) -> str:
    """
    Normalize labels for clustering/matching.

    This function should only remove superficial differences:
    - casing
    - Unicode variants
    - whitespace inconsistencies
    - punctuation formatting
    - legal identifier spacing

    It should NOT attempt semantic equivalence.
    """

    # Case normalization
    label = label.casefold()

    # Normalize Unicode representations
    label = unicodedata.normalize("NFKC", label)

    # Replace common invisible whitespace characters
    label = label.replace("\u00a0", " ")   # non-breaking space
    label = label.replace("\u202f", " ")   # narrow no-break space
    label = label.replace("\u200b", "")    # zero-width space

    # Normalize dash variants
    label = re.sub(
        r"[\u2010\u2011\u2012\u2013\u2014\u2212]",
        "-",
        label
    )

    # Collapse whitespace
    label = re.sub(r"\s+", " ", label)

    # Normalize numeric subdivisions:
    # Article 93 (3) -> Article 93(3)
    # Point 5 (1) -> Point 5(1)
    label = re.sub(
        r"(\d+)\s+\(\s*(\d+)\s*\)",
        r"\1(\2)",
        label
    )

    # Normalize alphabetic subdivisions:
    # Article 5 (a) -> Article 5(a)
    label = re.sub(
        r"(\d+)\s+\(\s*([a-z])\s*\)",
        r"\1(\2)",
        label
    )

    # Clean spaces inside parentheses:
    # ( EEC ) -> (eec)
    # ( 3 ) -> (3)
    label = re.sub(r"\(\s+", "(", label)
    label = re.sub(r"\s+\)", ")", label)

    return label.strip()

def has_identifier(text: str) -> bool:
    """
    Returns True if the label looks like a legal identifier
    or numbered designation.

    Conservative by design.
    """

    return bool(
        re.search(
            r"""
            \d                      # any digit
            |
            \([A-Za-z0-9]+\)        # (1), (a), (ii)
            |
            \b[IVXLCDM]+\b          # Roman numerals
            """,
            text,
            flags=re.IGNORECASE | re.VERBOSE,
        )
    )

def extract_uri_label(uri: URIRef):
    """
    Extract readable local name from a URI.

    Example:
        http://jurisynth/data/decision_no_30
            ->
        decision no 30
    """

    try:
        _, local = split_uri(uri)
    except Exception:
        local = str(uri).rsplit("/", 1)[-1]

    return local.replace("_", " ")


def prepare_document_resources(scored_triples):
    """
    Prepare custom resources for per-document deduplication.

    Returns
    -------
    document_resources

    {
        doc_id:
        {
            uri:
            {
                "uri": str,
                "label": str,
                "occurrences": [...],
                "contexts": set(),
                "neighbors": set()
            }
        }
    }
    """

    document_entities = defaultdict(dict)
    document_relations = defaultdict(dict)
    js_namespace = str(JS_DATA)

    for element in scored_triples:

        doc_id = element["doc_id"]
        chunk_id = element["chunk_id"]
        triple = element["triple"]

        for component in ("subject", "predicate", "object"):
            
            value = triple[component]

            if not isinstance(value, URIRef):
                continue
            if component != "predicate" and has_identifier(value):
                continue
            elif component == "predicate":
                target = document_relations
            else:
                target = document_entities
            
            uri = str(value)

            # Only deduplicate custom resources
            if not uri.startswith(js_namespace):
                continue

            if uri not in target[doc_id]:

                label = extract_uri_label(value)

                target[doc_id][uri] = {
                    "uri": uri,
                    "label": label,         # Original human-readable label
                    "occurrences": list(),  # Direct references into scored_triples
                    "contexts": set(),      # Optional future enrichment
                    "neighbors": set()      # Graph neighbours
                }

            target[doc_id][uri]["occurrences"].append({
                "triple_id": element["triple_id"],
                "chunk_id": chunk_id,
                "component": component
            })

    return document_entities, document_relations

In [215]:
document_entities, document_relations = prepare_document_resources(scored_triples)

In [216]:
class UnionFind:

    def __init__(self, n):
        self.parent = list(range(n))


    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(
                self.parent[x]
            )
        return self.parent[x]


    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)

        if ra != rb:
            self.parent[rb] = ra


def ngram_tokenize(text, n=3):
    text = text.lower().replace(" ", "")

    return [
        text[i:i+n]
        for i in range(len(text)-n+1)
    ]


# --------------------------------------------------
# Precompute embeddings globally
# --------------------------------------------------

def attach_resource_embeddings(
    document_resources,
    emb_model,
    batch_size=128
):
    """
    Compute embeddings for all resources across
    all documents.

    Adds:
        resource["embedding"]

    Parameters
    ----------
    document_resources : dict
        {
            doc_id:
                {
                    uri:
                        resource_dict
                }
        }

    """

    resource_entries = list()
    labels = list()


    for doc_id, resources in document_resources.items():
        for uri, resource in resources.items():
            resource_entries.append(
                (
                    doc_id,
                    uri,
                    resource
                )
            )

            labels.append(resource["label"])

    embeddings = emb_model.encode(
        labels,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    for (_, _, resource), embedding in zip(
        resource_entries,
        embeddings
    ):

        resource["embedding"] = embedding


# --------------------------------------------------
# Build clusters for one document
# --------------------------------------------------

from collections import defaultdict
import numpy as np


def build_candidate_clusters(
    doc_resources,
    similarity_threshold=0.9,
):
    """
    Build candidate equivalence clusters using
    pairwise cosine similarity.

    Parameters
    ----------
    doc_resources : dict

        Mapping:

        URIRef ->
        {
            "uri": URIRef,
            "label": str,
            "embedding": np.ndarray,
            "occurrences": list,
            "contexts": set,
            "neighbors": set
        }


    similarity_threshold : float
        Minimum cosine similarity required
        to merge two resources.


    Returns
    -------
    list[dict]

        [
            {
                "cluster_id": int,
                "resources": {
                    URIRef:
                        resource_dict
                }
            }
        ]
    """


    if not doc_resources:
        return list()


    uris = list(
        doc_resources.keys()
    )


    # ---------------------------------
    # Collect embeddings
    # ---------------------------------

    embeddings = np.stack(
        [
            doc_resources[uri]["embedding"]
            for uri in uris
        ]
    )


    # ---------------------------------
    # Pairwise cosine similarity
    #
    # embeddings are already normalized
    # ---------------------------------

    similarity_matrix = (
        embeddings @ embeddings.T
    )


    # ---------------------------------
    # Union-Find clustering
    # ---------------------------------

    uf = UnionFind(
        len(uris)
    )


    for i in range(len(uris)):

        for j in range(
            i + 1,
            len(uris)
        ):

            if (
                similarity_matrix[i, j]
                >= similarity_threshold
            ):

                uf.union(
                    i,
                    j
                )


    # ---------------------------------
    # Connected components
    # ---------------------------------

    grouped = defaultdict(list)


    for idx in range(len(uris)):

        grouped[
            uf.find(idx)
        ].append(idx)



    clusters = list()

    cluster_id = 1


    for members in grouped.values():

        # Ignore singleton resources
        if len(members) < 2:
            continue


        cluster_resources = dict()


        for idx in members:

            uri = uris[idx]

            cluster_resources[uri] = (
                doc_resources[uri]
            )


        clusters.append(
            {
                "cluster_id": cluster_id,
                "resources": cluster_resources
            }
        )

        cluster_id += 1


    return clusters

In [217]:
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

attach_resource_embeddings(
    document_entities,
    emb_model
)

attach_resource_embeddings(
    document_relations,
    emb_model
)

entity_clusters = dict()
relation_clusters = dict()

for doc_id, resources in document_entities.items():
    entity_clusters[doc_id] = build_candidate_clusters(resources)

for doc_id, resources in document_relations.items():
    relation_clusters[doc_id] = build_candidate_clusters(resources)

2026-07-19 11:35:32,961 - INFO - No device provided, using cpu
2026-07-19 11:35:33,817 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-19 11:35:33,837 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-19 11:35:34,210 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-19 11:35:34,231 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-19 11:35:34,235 - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-07-19 11

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-19 11:35:36,520 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-19 11:35:36,886 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-19 11:35:37,206 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-19 11:35:37,500 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-19 11:35:37,808 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-19 11:35:37,829 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/senten

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
# --------------------------------------------------
# Identifier detection
# --------------------------------------------------

IDENTIFIER_PATTERNS = [

    # Legal structural references
    r"\barticle\s+\d+",
    r"\bart\.\s*\d+",
    r"\bparagraph\s+\d+",
    r"\bpara\.\s*\d+",
    r"\bpoint\s+\(?[a-z0-9]+\)?",
    r"\bsection\s+\d+",
    r"\bchapter\s+[ivxlcdm\d]+",
    r"\btitle\s+[ivxlcdm\d]+",
    r"\bannex\s+[ivxlcdm\d]+",
    r"\brecital\s+\d+",


    # Legal instrument identifiers
    r"\bno\.?\s*\d+",
    r"\b\d+/\d+\b",


    # Directive / regulation style identifiers
    r"\b\d{4}/\d+\b",


    # Standalone year identifiers
    r"\b(19|20)\d{2}\b",


    # Explicit subdivisions
    r"\([a-z]\)",
    r"\(\d+\)",
]


IDENTIFIER_REGEX = [
    re.compile(
        pattern,
        flags=re.IGNORECASE
    )
    for pattern in IDENTIFIER_PATTERNS
]



def has_identifier(label: str) -> bool:
    """
    Returns True if a resource label contains an
    identifier-like component.

    Identifier-bearing entities are treated as
    unsafe for automatic deduplication because
    small differences may represent genuinely
    different legal concepts.

    Examples:

        Article 5
        Article 6
        Decision No 30/53
        Directive 2004/18/EC
        Annex III

    """

    if not label:
        return False


    label = label.strip()


    return any(
        regex.search(label)
        for regex in IDENTIFIER_REGEX
    )


# --------------------------------------------------
# Filter candidate clusters
# --------------------------------------------------

def filter_resolution_clusters(clusters):
    """
    Split candidate equivalence clusters into:

    review_clusters:
        Clusters that can proceed to semantic
        resolution.

    skipped_clusters:
        Clusters where every resource contains an
        identifier-like component and should not be
        merged automatically.


    Parameters
    ----------
    clusters : list[dict]

        Expected format:

        {
            "cluster_id": int,
            "resources": {
                URIRef:
                    {
                        "uri": URIRef,
                        "label": str,
                        ...
                    }
            }
        }


    Returns
    -------
    tuple[list, list]

        review_clusters,
        skipped_clusters

    """


    review_clusters = list()
    skipped_clusters = list()


    for cluster in clusters:

        labels = [
            resource["label"]
            for resource in cluster["resources"].values()
        ]

        # Conservative rule:
        # only skip when ALL resources are
        # identifier-bearing.
        if labels and all(
            has_identifier(label)
            for label in labels
        ):

            skipped_clusters.append(cluster)

        else:

            review_clusters.append(cluster)


    return (
        review_clusters,
        skipped_clusters
    )

# --------------------------------------------------
# Query Builder
# --------------------------------------------------

def build_resolution_query(cluster, cluster_type, doc_id):
    """
    Build an LLM resolution query for a candidate
    equivalence cluster.

    Parameters
    ----------
    cluster : dict
        Candidate cluster containing resources.

    cluster_type : str
        "entity" or "relation"
    
    doc_id : str
        Source document identifier.

    Returns
    -------
    str
        Structured query string for LLM resolution.
    """

    lines = [
        f"Document: {doc_id}",
        f"Resource type: {cluster_type}",
        f"Cluster ID: {cluster['cluster_id']}",
        "",
        "Candidate resources:"
    ]


    for idx, (uri, resource) in enumerate(
        cluster["resources"].items(),
        start=1
    ):

        lines.extend([
            f"{idx}.",
            f"URI: {uri}",
            f"Label: {resource['label']}",
            ""
        ])


    return "\n".join(lines)


In [ ]:
resolution_queries = list()
skipped_resolution_clusters = list()

for cluster_type, cluster_collection in [
    ("entity", entity_clusters),
    ("relation", relation_clusters)
]:

    for doc_id, clusters in cluster_collection.items():

        # ------------------------------------------
        # Filter identifier-heavy clusters
        # ------------------------------------------

        review_clusters, skipped_clusters = (
            filter_resolution_clusters(clusters)
        )

        skipped_resolution_clusters.extend(skipped_clusters)

        # ------------------------------------------
        # Build LLM queries
        # ------------------------------------------

        for cluster in review_clusters:

            resolution_queries.append(
                {
                    "doc_id": doc_id,
                    "cluster_type": cluster_type,
                    "cluster_id": cluster["cluster_id"],

                    # Keep original resources
                    "resources": cluster["resources"],

                    # Human-readable prompt
                    "query": build_resolution_query(
                        cluster,
                        cluster_type,
                        doc_id
                    )
                }
            )

entity_queries = [
    q for q in resolution_queries
    if q["cluster_type"] == "entity"
]

relation_queries = [
    q for q in resolution_queries
    if q["cluster_type"] == "relation"
]

In [289]:
def build_resolution_batches(
    queries,
    batch_size=10
):
    """
    Convert resolution queries into batched LLM requests.

    Each batch contains:
        - query: string sent to the LLM
        - lookup: mapping from
          (doc_id, cluster_type, cluster_id)
          to temporary resource ID -> URI

    Parameters
    ----------
    queries : list[dict]

        Expected format:

        {
            "doc_id": str,
            "cluster_type": str,
            "cluster_id": int,
            "resources": {
                uri: {
                    "label": str,
                    ...
                }
            },
            "query": str
        }

    batch_size : int
        Number of clusters per LLM request.

    Returns
    -------
    list[dict]
    """

    batches = list()

    for batch_start in range(
        0,
        len(queries),
        batch_size
    ):

        batch = queries[
            batch_start:
            batch_start + batch_size
        ]

        query_parts = list()
        lookup = dict()

        for cluster_query in batch:
            doc_id = cluster_query["doc_id"]
            cluster_type = cluster_query["cluster_type"]
            cluster_id = cluster_query["cluster_id"]

            key = (
                doc_id,
                cluster_type,
                cluster_id
            )

            resource_map = dict()

            for idx, (uri, resource) in enumerate(
                cluster_query["resources"].items(),
                start=1
            ):
                temp_id = f"r{idx}"
                resource_map[temp_id] = uri

            lookup[key] = resource_map
            query_parts.append(cluster_query["query"])

        batches.append(
            {
                "query": "\n\n".join(query_parts),
                "lookup": lookup
            }
        )

    return batches

In [290]:
entity_query_batches = build_resolution_batches(
    entity_queries,
    batch_size=10
)

relation_query_batches = build_resolution_batches(
    relation_queries,
    batch_size=10
)

In [291]:
resolution_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "cluster_id": {
                "type": "integer"
            },
            "resolutions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "canonical_label": {
                            "type": "string"
                        },
                        "members": {
                            "type": "array",
                            "items": {
                                "type": "string"
                            },
                            "minItems": 1
                        }
                    },
                    "required": [
                        "canonical_label",
                        "members"
                    ],
                    "additionalProperties": False
                }
            }
        },
        "required": [
            "cluster_id",
            "resolutions"
        ],
        "additionalProperties": False
    }
}

resolution_prompt = """
You are resolving potentially duplicate labels extracted from EU legal documents.

Your task is to decide whether labels within each cluster refer to the same resource.

Rules:

1. This is a clustering task, not a rewriting task.
- The canonical_label MUST be copied exactly from one of the provided members.
- Do not create, modify, or improve labels.

2. Each cluster must be partitioned correctly.
- Every input label must appear exactly once in the output.
- Do not assign a label to multiple groups.
- Do not omit any labels.

3. Merge labels only when they clearly refer to the same resource.
- If uncertain, keep labels separate.
- A cluster being provided does not mean all labels should be merged.


Example:

Cluster ID: 5
Resource type: entity

Candidates:

Resource ID: c5_r1
Label: the member

Resource ID: c5_r2
Label: a member

Output:
{
    "cluster_id": 5,
    "canonical": "c5_r1",
    "members": [
        "c5_r1",
        "c5_r2"
        ]
}

Now resolve the following clusters:

"""

In [297]:
# Parse outputs
resolved_entities = list()
resolved_relations = list()

def resolution_thread(batch):
    try:
        response = get_completion(
            system_prompt=resolution_prompt,
            query=batch["query"],
            schema=resolution_schema,
        )

        result = json.loads(response)

        # --------------------------------------------------
        # Attach lookup metadata
        # --------------------------------------------------
        #
        # The LLM only returns:
        #
        # {
        #     "cluster_id": 1,
        #     "resolutions": [...]
        # }
        #
        # Python restores:
        # doc_id, cluster_type, URI mappings
        #

        for cluster in result:
            cluster_id = cluster["cluster_id"]

            # Find corresponding lookup entry
            matches = [
                (
                    key,
                    resource_map
                )
                for key, resource_map in batch["lookup"].items()
                if key[2] == cluster_id
            ]

            if not matches:
                raise KeyError(f"No lookup found for cluster {cluster_id}")

            (
                key,
                resource_map
            ) = matches[0]

            doc_id, cluster_type, _ = key

            cluster["doc_id"] = doc_id
            cluster["cluster_type"] = cluster_type
            cluster["resource_map"] = resource_map

        return result

    except Exception as e:
        print(f"Resolution failed: {e}")
        return list()

with ThreadPoolExecutor(max_workers=20) as executor:

    resolved_entities = [
        item
        for batch_result in executor.map(
            resolution_thread,
            entity_query_batches
        )
        for item in batch_result
    ]

    resolved_relations = [
        item
        for batch_result in executor.map(
            resolution_thread,
            relation_query_batches
        )
        for item in batch_result
    ]

2026-07-19 14:09:22,007 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 14:09:22,844 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 14:09:25,347 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 14:09:25,390 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 14:09:25,489 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 14:09:25,581 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 14:09:25,691 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 14:09:25,726 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-19 14:09

In [298]:
resolved_entities[:30]

[{'cluster_id': 1,
  'resolutions': [{'canonical_label': 'http://jurisynth/data/high_authority',
    'members': ['http://jurisynth/data/the_high_authority',
     'http://jurisynth/data/high_authority']}],
  'doc_id': '31953D0030en.md',
  'cluster_type': 'entity',
  'resource_map': {'r1': 'http://jurisynth/data/the_high_authority',
   'r2': 'http://jurisynth/data/high_authority'}},
 {'cluster_id': 1,
  'resolutions': [{'canonical_label': 'http://jurisynth/data/element_of_control_of_an_undertaking',
    'members': ['http://jurisynth/data/element_of_control_of_an_undertaking']},
   {'canonical_label': 'http://jurisynth/data/control_of_an_undertaking',
    'members': ['http://jurisynth/data/control_of_an_undertaking']}],
  'doc_id': '31953D0030en.md',
  'cluster_type': 'entity',
  'resource_map': {'r1': 'http://jurisynth/data/the_high_authority',
   'r2': 'http://jurisynth/data/high_authority'}},
 {'cluster_id': 2,
  'resolutions': [{'canonical_label': 'http://jurisynth/data/rights',
    '

In [300]:
resolved_entities[0]

{'cluster_id': 1,
 'resolutions': [{'canonical_label': 'http://jurisynth/data/high_authority',
   'members': ['http://jurisynth/data/the_high_authority',
    'http://jurisynth/data/high_authority']}],
 'doc_id': '31953D0030en.md',
 'cluster_type': 'entity',
 'resource_map': {'r1': 'http://jurisynth/data/the_high_authority',
  'r2': 'http://jurisynth/data/high_authority'}}

In [ ]:
# ---------------------------------------------------------------------
# Build resolution maps from LLM outputs
# ---------------------------------------------------------------------

def build_resolution_map(
    resolved_clusters
):
    """
    Convert LLM cluster resolutions into:

        old_uri -> canonical_uri

    Parameters
    ----------
    resolved_clusters : list[dict]
        Enriched LLM outputs containing:

        {
            "cluster_id": int,
            "resolutions": [...],
            "resource_map": {
                temporary_id: URI
            }
        }

    Returns
    -------
    dict
        URI replacement map.
    """

    resolution_map = dict()

    for cluster in resolved_clusters:
        for resolution in cluster["resolutions"]:
            canonical_uri = resolution["canonical_label"]

            for member_uri in resolution["members"]:
                resolution_map[member_uri] = canonical_uri

    return resolution_map


# ---------------------------------------------------------------------
# Apply resolutions to triples
# ---------------------------------------------------------------------

def apply_resolution(
    scored_triples,
    entity_map,
    relation_map
):
    """
    Apply URI resolution maps to scored triples.

    Parameters
    ----------
    scored_triples : list[dict]

    Returns
    -------
    list[dict]
    """

    resolved_triples = list()

    for element in scored_triples:
        updated = element.copy()
        triple = element["triple"].copy()

        # -----------------------------
        # Resolve all components
        # -----------------------------

        if (
            isinstance(triple["subject"], URIRef)
            and triple["subject"] in entity_map
        ):
            triple["subject"] = entity_map[triple["subject"]]

        if (
            isinstance(triple["object"], URIRef)
            and triple["object"] in entity_map
        ):
            triple["object"] = entity_map[triple["object"]]

        if (
            isinstance(triple["predicate"], URIRef)
            and triple["predicate"] in relation_map
        ):
            triple["predicate"] = relation_map[triple["predicate"]]

        updated["triple"] = triple
        resolved_triples.append(updated)

    return resolved_triples


# ---------------------------------------------------------------------
# Generate URI resolution maps
# ---------------------------------------------------------------------

entity_map = build_resolution_map(
    resolved_entities
)

relation_map = build_resolution_map(
    resolved_relations
)

# ---------------------------------------------------------------------
# Apply resolutions
# ---------------------------------------------------------------------

resolved_triples = apply_resolution(
    scored_triples,
    entity_map,
    relation_map
)

### Graph Serializer

In [100]:
KNOWN_PREDICATES = {
    "rdf:type": RDF.type,
    "type": RDF.type,
    "rdfs:label": RDFS.label,
    "label": RDFS.label,
    "owl:sameAs": OWL.sameAs
}

def normalize_uri(text):
    text = text.strip()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^\w\-\#]", "", text)  # keep #
    return text

def resolve_resource(p):
    # normalize
    key = p.strip()

    # ✅ case 1: known RDF predicate
    for known in KNOWN_PREDICATES.keys():
        if key.lower() == known.lower():
            key = known
            return KNOWN_PREDICATES[key]

    # ✅ case 2: in your ontology dict
    for resource in rdf_dict.keys():
        if key.lower() == resource.lower():
            key = resource
            return URIRef(rdf_dict[key])

    # ✅ case 3: fallback → your namespace
    return JS[key.replace(" ", "_")]

In [ ]:
pure_triples = {filename: set() for filename in cleaned_data.keys()}
errors = {filename: list() for filename in cleaned_data.keys()}

JS = Namespace("http://jurisynth.org/cdmext/data/")

for filename, triple_list in copy.deepcopy(cleaned_data).items():
    for triple in triple_list:

        subj = normalize_uri(triple["subject"]["name"])
        obj = normalize_uri(triple["object"]["name"])

        pred_labels = triple["predicate"].get("labels", [])

        if pred_labels:
            for pred_tag in pred_labels:
                try:
                    pred = resolve_resource(pred_tag)

                    pure_triples[filename].add((JS[subj], pred, JS[obj]))

                except Exception:
                    errors[filename].append((JS[subj], pred_tag, JS[obj], 1))

        else:
            key = triple["predicate"]["name"]

            pred = resolve_resource(key)

            pure_triples[filename].add((JS[subj], pred, JS[obj]))

total = sum([len(triple_list) for triple_list in pure_triples.values()])
print("No. of proper triples:", total)

subj_tags = copy.deepcopy([
    (triple['subject']["name"], tuple(triple['subject']["labels"]), filename)
    for filename, triple_list in cleaned_data.items()
    for triple in triple_list
    ])
obj_tags = copy.deepcopy([
    (triple['object']["name"], tuple(triple['object']["labels"]), filename)
    for filename, triple_list in cleaned_data.items()
    for triple in triple_list
    ])
triple_tags = set(subj_tags + obj_tags)

empty_tags = set([pair for pair in triple_tags if len(pair[1]) == 0])
triple_tags -= empty_tags

for entity, label_list, filename in triple_tags:
    norm_entity = normalize_uri(entity)
    for label in label_list:
        clean_label = normalize_uri(strip_cdm_prefix(label))
        resolved_label = resolve_resource(clean_label)

        pure_triples[filename].add((
            JS[norm_entity],
            RDF.type,
            resolved_label
            ))

total = sum([len(triple_list) for triple_list in errors.values()])
print()
print("No. of erroneous triples:", total)

No. of proper triples: 4056

No. of erroneous triples: 0


*Issue (Future Work):*
* Route highly-matched resources to their appropriate namespaces.

In [103]:
pure_triples

{'31961D0408_01_en.md': {(rdflib.term.URIRef('http://jurisynth.org/cdmext/data/1961-03-08'),
   rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
   rdflib.term.URIRef('http://purl.org/dc/terms/date')),
  (rdflib.term.URIRef('http://jurisynth.org/cdmext/data/Commission_Decision_on_modification_of_aid_system_in_Italy'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/addresses'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/compatibility_of_Italian_draft_law_with_common_market_under_Article_923c')),
  (rdflib.term.URIRef('http://jurisynth.org/cdmext/data/Commission_Decision_on_modification_of_aid_system_in_Italy'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/date_adopted'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/1961-03-08')),
  (rdflib.term.URIRef('http://jurisynth.org/cdmext/data/Commission_Decision_on_modification_of_aid_system_in_Italy'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/issued_by'),
   rdfli

In [104]:
errors_list = list()

for err_list in errors.values():
    errors_list.extend(err_list)

len(errors_list)

0

*Handling Erroneous Triples (if any):*

In [140]:
def batch_list(lst, size=10):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

def error_fix_batch(errors):
    batch_prompt = "Fix the labels for the following triples.\n\n"

    metadata = []

    for i, error in enumerate(errors):
        err_index = error[-1]
        incomplete_tag = error[err_index]
        
        if "#" not in incomplete_tag:
            continue  # or handle differently

        reference = incomplete_tag.split("#")[-1]
        triple = ", ".join(error[:3])
        query_list = [key for key in rdf_dict.keys() if reference in key]

        batch_prompt += f"""
        Item {i}:
        Triple: {triple}
        Incorrect label: {incomplete_tag}
        Candidates: {", ".join(query_list)}
        """

        metadata.append((i, error, err_index))

    batch_prompt += """
    Return the corrected labels as a JSON array of strings in order:
    ["label1", "label2", ...]
    Only return the JSON array.
    """

    # 🔑 ONE API CALL
    labels = json.loads(get_completion(system_prompt=batch_prompt, query=""))

    corrected = []

    for label, (_, error, err_index) in zip(labels, metadata):
        triple = list(error[:3])
        try:
            triple[err_index] = rdf_dict[label]
        except:
            print(triple, err_index)
        trp_dict = {"triple": triple, "filename": filename}
        corrected.append(triple)

    return corrected

In [ ]:
error_batches = list(batch_list(errors_list, 10))

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(error_fix_batch, error_batches))

fixed_list = []
for batch in results:
    fixed_list.extend(batch)

In [105]:
named_graph = Dataset()
SOURCE = Namespace("http://jurisynth.org/cdmext/source/")

named_graph.bind("js", JS)
named_graph.bind("source", SOURCE)

for idx, ns in schema_namespaces:
    named_graph.bind(idx, ns)

for filename, triple_list in pure_triples.items():

    graph_uri = SOURCE[normalize_uri(filename)]
    g = named_graph.graph(graph_uri)

    for s, p, o in triple_list:
        g.add((s, p, o))
        
named_graph.serialize("eu_legislation_graph.nq", format="nquads")

<Graph identifier=N65eac2211c334231b44be1f3bfb8745d (<class 'rdflib.graph.Dataset'>)>

### Schema Validator (FUTURE WORK)